# DataMate — Kaggle GPU smoke test

Mục tiêu: xác nhận pipeline chạy được với LLM thật trên GPU Kaggle (T4 x2).
Đây **không** phải bài đo 6 giờ; máy chấm là 2 x RTX 5090.

Cài đặt notebook trước khi chạy (Settings bên phải):
- Accelerator: **GPU T4 x2**
- Internet: **On** (chỉ cần cho cell cài đặt / clone; tắt được ở cell suy luận)
- Persistence: Files only
- Input: add dataset chứa trọng số `Qwen2.5-Coder-7B-Instruct-AWQ` (xem cell 2 nếu chưa có)

In [ ]:
# Cell 1 — lấy source code (đổi BRANCH nếu P0 nằm trên nhánh khác main)
import os, subprocess, pathlib
REPO_URL = "https://github.com/HoL0ngg/Zalo-AI-Challenge-2026.git"
BRANCH = "main"
ROOT = pathlib.Path("/kaggle/working/datamate")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
# Phải thấy commit P0 (có file src/datamate/codegen/sql_parser.py), nếu không là đang chạy code cũ.
assert (ROOT / "src/datamate/codegen/sql_parser.py").exists(), "Chưa push code P0 lên GitHub"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Cell 2 — trọng số mô hình.
# Lần đầu: tải từ Hugging Face (Internet On), rồi Save Version -> Output -> "New Dataset"
# để các lần sau add làm Input và chạy offline.
# Tự tìm thư mục chứa config.json + *.safetensors dưới /kaggle/input (Kaggle có thể đặt
# dataset ở /kaggle/input/<slug>/ hoặc /kaggle/input/datasets/<user>/<slug>/, có hoặc không có thư mục con).
import pathlib
MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"

def find_weights(root="/kaggle/input"):
    hits = []
    for cfg in pathlib.Path(root).rglob("config.json"):
        folder = cfg.parent
        if any(folder.glob("*.safetensors")) and (folder / "tokenizer.json").exists():
            hits.append(folder)
    return sorted(hits, key=lambda d: len(str(d)))

found = find_weights()
if found:
    weights = found[0]
    print("Dùng trọng số có sẵn:", weights)
    if len(found) > 1:
        print("Các thư mục khác cũng khớp:", found[1:])
else:
    weights = pathlib.Path("/kaggle/working/Qwen2.5-Coder-7B-Instruct-AWQ")
    if not (weights / "config.json").exists():
        from huggingface_hub import snapshot_download
        snapshot_download(MODEL_ID, local_dir=str(weights))
    print("Đã tải trọng số về:", weights, "— hãy Save Version và tạo Dataset từ Output")

link = ROOT / "models" / "qwen-coder-awq"          # phải khớp local_path trong configs/models_kaggle.yaml
link.parent.mkdir(exist_ok=True)
if link.is_symlink() or link.exists():
    link.unlink()
link.symlink_to(weights, target_is_directory=True)
assert (link / "config.json").exists(), f"Symlink trọng số chưa đúng: {link} -> {weights}"
print("models/qwen-coder-awq ->", link.resolve())
print(sorted(f.name for f in weights.iterdir())[:20])

In [ ]:
# Cell 3 — cài phụ thuộc (5–10 phút). vLLM kéo theo bản torch của riêng nó (hiện là cu130).
%pip install -q -e ".[test]"
%pip install -q rapidfuzz Unidecode sqlglot
%pip install -q vllm
# Kaggle cài sẵn torchaudio/torchvision build cho CUDA cũ hơn; lệch CUDA với torch mới sẽ làm
# transformers crash khi import. Dự án không dùng audio nên gỡ torchaudio; torchvision chỉ giữ
# nếu đúng bản CUDA của torch.
%pip uninstall -q -y torchaudio
import sys
import torch
# torchvision phải import được VÀ khớp bản torch: vLLM >= 0.32 import torchvision ở bước warmup kernel
# (thiếu là "No module named 'torchvision'" khi khởi động engine). Không so chuỗi version CUDA:
# torchvision.version.cuda = "13000" còn torch.version.cuda = "13.0" nên luôn lệch.
def torchvision_ok():
    import importlib
    try:
        for name in [m for m in sys.modules if m == "torchvision" or m.startswith("torchvision.")]:
            del sys.modules[name]
        importlib.import_module("torchvision.ops")   # nạp extension C++, lỗi nếu lệch torch/CUDA
        from torchvision.transforms import InterpolationMode  # noqa: F401
        return True
    except Exception as exc:  # noqa: BLE001
        print("torchvision không dùng được:", repr(exc)[:200])
        return False

if not torchvision_ok():
    minor = int(torch.__version__.split(".")[1])          # torch 2.N <-> torchvision 0.(N+15)
    tag = "cu" + torch.version.cuda.replace(".", "")
    !{sys.executable} -m pip uninstall -q -y torchvision
    !{sys.executable} -m pip install -q --no-deps "torchvision==0.{minor + 15}.*" --index-url https://download.pytorch.org/whl/{tag}
assert torchvision_ok(), "torchvision vẫn lỗi: vLLM sẽ không khởi động được engine"
import torchvision
print("torchvision", torchvision.__version__)
import vllm
print("vllm", vllm.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
import transformers.audio_utils  # phải import được mà không lỗi
print("transformers", transformers.__version__, "OK")

In [ ]:
# Cell 4 — kiểm tra CPU/mock trước khi đụng GPU
!python -m pytest -q
!python -m datamate.cli run --data-dir data/samples --result-dir /kaggle/working/result/sample
!python -m datamate.cli eval /kaggle/working/result/sample/submission.jsonl data/samples/answers.jsonl

In [ ]:
# Cell 5 — chạy LLM thật trên tập train (có answers.jsonl để chấm). Từ cell này có thể tắt Internet.
import os
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
                  VLLM_NO_USAGE_STATS="1", PYTHONHASHSEED="20261030")
import time; _t0 = time.time()
!python -m datamate.cli run --config configs/kaggle.yaml     --data-dir dataset/train --result-dir /kaggle/working/result/train
WALL_1GPU = time.time() - _t0
import pathlib
sub = pathlib.Path("/kaggle/working/result/train/submission.jsonl")
assert sub.exists() and sub.stat().st_size > 0, "Chạy LLM lỗi, không có submission: đọc traceback ở trên (thường là vLLM không khởi động)"
open("/kaggle/working/wall_1gpu.txt", "w").write(str(WALL_1GPU))
print(f"1 GPU: {WALL_1GPU:.0f}s")

In [ ]:
# Cell 6 — chấm điểm train + báo cáo theo bước (SỐ BASELINE: gửi file report_train.json)
R = "/kaggle/working/result/train"
META = "synth/out/.debug/train/questions_meta.jsonl"
!python -m datamate.cli validate {R}/submission.jsonl
!python -m datamate.cli eval {R}/submission.jsonl dataset/train/answers.jsonl
!python -m datamate.cli report {R} dataset/train/answers.jsonl --meta {META} > /kaggle/working/report_train.json
!head -c 2500 /kaggle/working/report_train.json

In [ ]:
# Cell 7 (tuỳ chọn) — chạy lần 2 để kiểm tra tính tái lập (yêu cầu lệch <= 0.03)
!python -m datamate.cli run --config configs/kaggle.yaml     --data-dir dataset/train --result-dir /kaggle/working/result/train_rerun
!python -m datamate.cli eval /kaggle/working/result/train_rerun/submission.jsonl dataset/train/answers.jsonl
!diff <(sort /kaggle/working/result/train/submission.jsonl) <(sort /kaggle/working/result/train_rerun/submission.jsonl) | head -20

In [ ]:
# Cell 8 (tuỳ chọn, nên chạy) — dev: đáp án nằm ở synth/out/.debug/dev, dùng để đo ngoài tập đã nhìn khi chỉnh prompt
R = "/kaggle/working/result/dev"
!python -m datamate.cli run --config configs/kaggle.yaml     --data-dir dataset/dev --result-dir {R}
!python -m datamate.cli validate {R}/submission.jsonl
!python -m datamate.cli eval {R}/submission.jsonl synth/out/.debug/dev/answers.jsonl
!python -m datamate.cli report {R} synth/out/.debug/dev/answers.jsonl --meta synth/out/.debug/dev/questions_meta.jsonl > /kaggle/working/report_dev.json
!head -c 2500 /kaggle/working/report_dev.json

## 2 GPU (data parallel)
Mỗi GPU một tiến trình `datamate run` với cùng model (`tensor_parallel_size: 1`), mỗi tiến trình lo một nhóm business rời nhau (`src/datamate/runtime/dataparallel.py`). Vẫn là pipeline tuần tự trong từng tiến trình, nên đây là cách rẻ nhất để dùng cả hai card; kết quả được gộp lại đúng thứ tự `questions.jsonl`. Chạy sau cell 5 để có mốc 1 GPU so sánh thời gian.

In [ ]:
# Cell 8a — kiểm tra có đủ 2 GPU
import torch
n_gpu = torch.cuda.device_count()
print("GPU nhìn thấy:", n_gpu, [torch.cuda.get_device_name(i) for i in range(n_gpu)])
assert n_gpu >= 2, "Bật Accelerator = GPU T4 x2 trong Settings"

In [ ]:
# Cell 8b — train trên 2 GPU, chấm và báo cáo (gửi report_train_2gpu.json)
import os, time
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", VLLM_NO_USAGE_STATS="1", PYTHONHASHSEED="20261030")
R2 = "/kaggle/working/result/train_2gpu"
_t0 = time.time()
!python -m datamate.runtime.dataparallel --config configs/kaggle.yaml     --data-dir dataset/train --result-dir {R2} --gpus 0,1
WALL_2GPU = time.time() - _t0
import json as _json, pathlib
_sub = pathlib.Path(R2) / "submission.jsonl"
assert _sub.exists() and _sub.stat().st_size > 0, f"Không có kết quả: đọc {R2}/.shards/shard_*.log"
_logs = sorted(pathlib.Path(R2, ".shards").glob("shard_*.log"))
_bad = [p.name for p in _logs if "Traceback" in p.read_text(errors="ignore")]
assert not _bad, f"Shard lỗi: {_bad} — đọc log trong {R2}/.shards/"
try:
    w1 = float(open("/kaggle/working/wall_1gpu.txt").read())
    print(f"1 GPU {w1:.0f}s | 2 GPU {WALL_2GPU:.0f}s | tăng tốc x{w1 / WALL_2GPU:.2f} (gồm cả thời gian nạp model)")
except FileNotFoundError:
    print(f"2 GPU {WALL_2GPU:.0f}s (chưa có mốc 1 GPU: chạy cell 5 trước)")
!python -m datamate.cli validate {R2}/submission.jsonl
!python -m datamate.cli eval {R2}/submission.jsonl dataset/train/answers.jsonl
!python -m datamate.cli report {R2} dataset/train/answers.jsonl --meta synth/out/.debug/train/questions_meta.jsonl > /kaggle/working/report_train_2gpu.json
!head -c 1500 /kaggle/working/report_train_2gpu.json
# Nếu một shard lỗi: đọc {R2}/.shards/shard_*.log

In [ ]:
# Cell 8c (tuỳ chọn) — so 1 GPU và 2 GPU: câu trả lời phải giống nhau (cùng seed, temperature 0)
import json
def load(p): return {r["id"]: r for r in map(json.loads, open(p, encoding="utf-8"))}
one, two = load("/kaggle/working/result/train/submission.jsonl"), load(R2 + "/submission.jsonl")
diff = [i for i in one if one[i] != two.get(i)]
print(f"khác nhau: {len(diff)}/{len(one)}", diff[:10])

## Đo thêm
- **Số token thật của prompt** (cần tokenizer của model, chạy sau cell 2):
  `!python scripts/measure_prompts.py --data-dir dataset/train --tokenizer models/qwen-coder-awq`
- **So với trước P0** (tuỳ chọn): chạy 2 cell tiếp theo.

In [ ]:
# Cell 9 — baseline TRƯỚC P0 (commit 5376138) để biết P0 có thực sự giúp không. Dùng lại trọng số ở cell 2.
BASE_SHA = "5376138"
BASE = pathlib.Path("/kaggle/working/datamate_base")
if not BASE.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(BASE)], check=True)
subprocess.run(["git", "-C", str(BASE), "checkout", BASE_SHA], check=True)
(BASE / "models").mkdir(exist_ok=True)
base_link = BASE / "models" / "qwen-coder-awq"
if base_link.is_symlink() or base_link.exists():
    base_link.unlink()
base_link.symlink_to(weights, target_is_directory=True)
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", VLLM_NO_USAGE_STATS="1", PYTHONHASHSEED="20261030")
# PYTHONPATH đứng trước bản editable, nên chạy đúng code cũ trong BASE.
!cd {BASE} && PYTHONPATH={BASE}/src python -m datamate.cli run --config configs/kaggle.yaml     --data-dir dataset/train --result-dir /kaggle/working/result/train_preP0

In [ ]:
# Cell 10 — chấm bản trước P0 bằng công cụ báo cáo mới (đọc được log cũ)
R0 = "/kaggle/working/result/train_preP0"
!python -m datamate.cli eval {R0}/submission.jsonl dataset/train/answers.jsonl
!python -m datamate.cli report {R0} dataset/train/answers.jsonl --meta synth/out/.debug/train/questions_meta.jsonl > /kaggle/working/report_train_preP0.json
!head -c 2500 /kaggle/working/report_train_preP0.json

In [ ]:
# Cell 11 — đóng gói kết quả để tải về (Output -> datamate_results.zip) và gửi cho mình
!cd /kaggle/working && zip -rq datamate_results.zip report_*.json result -x "result/sample/*"
!ls -la /kaggle/working/datamate_results.zip